# TESSERACT v1.1 — Kaggle Training & Stage 2 Fine-Tuning

**Prerequisites (Right Sidebar)**:
1. **Settings → Accelerator**: GPU T4 ×2 (or P100)
2. **+ Add Input**:
   - Code dataset (contains `tesseract_v1_patched.py`)
   - **DASVO TartanAir RGB-D Validation Split** (`pandrii000/dasvo-tartanair-rgb-d-validation-split`)
   - *(Optional for Stage 2)*: Checkpoint dataset (contains `checkpoint.pt` at Epoch 24)

### Two-Stage Training Protocol
- **Stage 1 (Epochs 0–24)**: Standard cross-environment pre-training under canonical TartanAir virtual pinhole sensor.
- **Stage 2 (Fine-Tuning, 15 Epochs)**: Camera-invariant fine-tuning using **Dynamic Pinhole Intrinsics Crop Augmentation** ($L \in [0.55\cdot S_{\max}, S_{\max}]$, FOV $\sim 45^\circ$ to $74^\circ$, $\text{LR}=5\times 10^{-5}$).

In [ ]:
# [1] Verify GPU accelerator allocation
!nvidia-smi
import torch
print("CUDA available :", torch.cuda.is_available())
print("Device count   :", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"Device {i}       : {torch.cuda.get_device_name(i)}")

In [ ]:
# [2] Locate script, verify dataset mount, and prepare execution environment
import glob, os, shutil

scripts = sorted(glob.glob('/kaggle/input/**/tesseract_v1_patched.py', recursive=True))
if not scripts:
    scripts = sorted(glob.glob('/kaggle/input/**/tesseract.py', recursive=True))
assert scripts, 'tesseract_v1_patched.py not found — please attach the code dataset!'
input_script = scripts[-1]
print(f"Found mounted script: {input_script}")

# Ensure writable working copy
working_script = "/kaggle/working/tesseract_v1_patched.py"
shutil.copyfile(input_script, working_script)
SCRIPT = working_script
os.environ["SCRIPT_PATH"] = SCRIPT

# Verify TartanAir mount
def list_mounts(base='/kaggle/input'):
    out = []
    if not os.path.isdir(base): return out
    for name in sorted(os.listdir(base)):
        p = os.path.join(base, name)
        if not os.path.isdir(p): continue
        if name == 'datasets':
            for owner in sorted(os.listdir(p)):
                op = os.path.join(p, owner)
                if os.path.isdir(op):
                    for slug in sorted(os.listdir(op)):
                        if os.path.isdir(os.path.join(op, slug)):
                            out.append(f'datasets/{owner}/{slug}')
        else:
            out.append(name)
    return out

mounts = list_mounts()
print('Mounted inputs:', mounts)
assert any(('tartanair' in m.lower()) or ('dasvo' in m.lower()) for m in mounts), (
    'The DASVO TartanAir dataset is NOT attached to this notebook!\n'
    'Right sidebar → Input → "+ Add Input" → search "dasvo-tartanair-rgb-d-validation-split"'
)
print('Ready to run ✓')

In [ ]:
# [3] Sanity checks: config summary, smoke test, and 16 unit tests
!python "$SCRIPT_PATH" --config
!python "$SCRIPT_PATH" --smoke
!python "$SCRIPT_PATH" --test

In [ ]:
# [4A] OPTION A: Launch Standard Training (Epochs 0–30) / Resume
# Use this cell to train from scratch or resume an ongoing Stage 1 run.
# !python "$SCRIPT_PATH" \
#     --train auto \
#     --ablation full \
#     --split-mode cross_env \
#     --batch-size 4 \
#     --epochs 30 \
#     --output-dir /kaggle/working/outputs

In [ ]:
# [4B] OPTION B: STAGE 2 — Camera-Invariant Pinhole Fine-Tuning
import os, glob

# Guard: self-heal SCRIPT_PATH if Cell [2] was skipped
if "SCRIPT_PATH" not in os.environ or not os.path.isfile(os.environ.get("SCRIPT_PATH", "")):
    scripts = sorted(glob.glob("/kaggle/input/**/tesseract_v1_patched.py", recursive=True)) + \
              sorted(glob.glob("/kaggle/working/tesseract_v1_patched.py"))
    if scripts:
        os.environ["SCRIPT_PATH"] = scripts[-1]
        print(f"Set SCRIPT_PATH -> {os.environ["SCRIPT_PATH"]}")
    else:
        raise FileNotFoundError("tesseract_v1_patched.py not found! Please attach code dataset.")

# Instant lookup: prunes TartanAir directory from search to prevent slow network FUSE traversal
ckpt_candidates = []
if os.path.isfile("/kaggle/working/outputs/checkpoint.pt"):
    ckpt_candidates.append("/kaggle/working/outputs/checkpoint.pt")

for root, dirs, files in os.walk("/kaggle/input"):
    # Prune TartanAir directories so search finishes in 0.05s instead of 15 minutes!
    if any(k in root.lower() for k in ("tartanair", "dasvo", "image_left", "depth_left", "p00")):
        dirs.clear()
        continue
    for f in files:
        if f.endswith(".pt") or ("checkpoint" in f.lower() and f.endswith((".zip", ".pt"))):
            ckpt_candidates.append(os.path.join(root, f))

print("Found checkpoint candidate(s):", ckpt_candidates)
assert ckpt_candidates, "No checkpoint found! Please check attached inputs in right sidebar."
base_ckpt = ckpt_candidates[0]
os.environ["BASE_CKPT"] = base_ckpt
print(f"Initializing Stage 2 fine-tuning from: {base_ckpt}")

!python -u "$SCRIPT_PATH" \
    --train auto \
    --finetune "" \n    --epochs 15 \
    --batch-size 4 \
    --finetune "" \n    --pinhole-aug \
    --pinhole-min-scale 0.55 \
    --output-dir /kaggle/working/outputs

In [ ]:
# [5] Inspect generated outputs & show latest validation depth prediction
import os, glob
from IPython.display import Image, display

out_dir = "/kaggle/working/outputs"
if os.path.isdir(out_dir):
    print("Output directory contents:")
    for f in sorted(os.listdir(out_dir)):
        sz = os.path.getsize(os.path.join(out_dir, f))
        print(f"  {f:<30} ({sz / (1024*1024):.2f} MB)")

    viz_files = sorted(glob.glob(f"{out_dir}/viz/*.png"))
    if viz_files:
        print(f"\nDisplaying latest validation render: {viz_files[-1]}")
        display(Image(filename=viz_files[-1]))
    else:
        print("No visualization images generated yet.")
else:
    print("No output directory found yet.")

In [ ]:
# [6] Quantitative Evaluation (Runs if checkpoint exists)
# Computes AbsRel, SqRel, RMSE, and Delta thresholds on held-out test environments.
import os
ckpt = "/kaggle/working/outputs/checkpoint.pt"
if os.path.isfile(ckpt):
    !python "$SCRIPT_PATH" --evaluate "$ckpt" --train auto
else:
    print(f"Checkpoint not found at {ckpt}. Training must complete at least 1 epoch first.")

In [ ]:
# [7] Multi-FOV Synthetic Camera Benchmark
# Sweeps horizontal field-of-view (50° to 100°) on test frames to demonstrate
# camera-grounded conditioning and metric scale preservation across focal lengths.
import glob, os
from IPython.display import Image, display

imgs = sorted(glob.glob('/kaggle/input/**/image_left/*.png', recursive=True))
ckpt = "/kaggle/working/outputs/checkpoint.pt"

if imgs and os.path.isfile(ckpt):
    test_img = imgs[0]
    print(f"Running Multi-FOV benchmark on test image: {test_img}")
    !python "$SCRIPT_PATH" \
        --eval-multi-fov "$test_img" \
        --resume "$ckpt" \
        --output-dir /kaggle/working/outputs

    sweep_plots = sorted(glob.glob("/kaggle/working/outputs/*_multi_fov_sweep.png"))
    if sweep_plots:
        print(f"\nDisplaying Multi-FOV sweep visualization: {sweep_plots[-1]}")
        display(Image(filename=sweep_plots[-1]))
else:
    print("Test images or checkpoint not found.")

In [ ]:
# [8] Single Image Depth Prediction & 3D Point Cloud Export
import glob, os

imgs = sorted(glob.glob('/kaggle/input/**/image_left/*.png', recursive=True))
ckpt = "/kaggle/working/outputs/checkpoint.pt"
if imgs and os.path.isfile(ckpt):
    test_img = imgs[0]
    print(f"Predicting depth for test image: {test_img}")
    !python "$SCRIPT_PATH" \
        --predict "$test_img" \
        --resume "$ckpt" \
        --export-ply /kaggle/working/outputs/predicted_mesh.ply \
        --output-dir /kaggle/working/outputs
else:
    print("Test images or checkpoint not found.")

In [ ]:
# [9] Package all results for easy download from the right-hand panel
!cd /kaggle/working && zip -r tesseract_outputs.zip outputs/ -x "*.pyc"
print("Zipped all outputs to /kaggle/working/tesseract_outputs.zip ✓")